In [15]:
import pandas as pd
import os
import re
# 1일차의 데이터 로드 및 데이터 정제

data_path = "../../data/Mobile Reviews Sentiment.csv" 

df = pd.read_csv(data_path)

# 텍스트 정제
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df['cleaned_review'] = df['review_text'].apply(clean_text)


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# TF-IDF Vectorizer 선언 (너무 흔하거나 너무 희귀한 단어는 제외하여 노이즈 줄이기)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# 텍스트 데이터를 TF-IDF 행렬로 변환
X = tfidf.fit_transform(df['cleaned_review'])

# 타겟 레이블 설정 (여기서는 기존 sentiment를 이진 분류 형태로 매핑하거나 활용)
# 만약 sentiment 컬럼이 문자열('Positive', 'Negative' 등)이라면 0과 1로 변환해 줍니다.
y = df['sentiment'].apply(lambda x: 1 if str(x).lower() in ['positive', 'good', '1'] else 0)

# Train / Test 데이터셋 분할 (학습용 80%, 테스트용 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("--- Train & Test Set Shape ---")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

--- Train & Test Set Shape ---
X_train shape: (40000, 433), X_test shape: (10000, 433)


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 로지스틱 회귀 모델 선언
model = LogisticRegression(max_iter=1000, random_state=42)

# 모델 학습
model.fit(X_train, y_train)

# 예측 수행
y_pred = model.predict(X_test)

# 성능 평가 (Accuracy 및 F1-Score)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

# 출력
print("--- Model Performance ---")
print(f"Accuracy: {acc:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

--- Model Performance ---
Accuracy: 1.0000
F1-Score: 1.0000

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4536
           1       1.00      1.00      1.00      5464

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000



precision    recall  f1-score 전부 1.00 은 말이안됨
평점 기반으로 레이블을 재정의 해보아야 할듯

In [18]:
# TF-IDF 피처 이름(단어들)과 모델의 계수(Coefficient) 매핑
feature_names = tfidf.get_feature_names_out()
coefficients = model.coef_[0]

# 데이터프레임으로 묶기
coef_df = pd.DataFrame({'word': feature_names, 'coefficient': coefficients})

# 긍정에 큰 영향을 주는 단어 Top 10 vs 부정에 큰 영향을 주는 단어 Top 10 확인
top_positive = coef_df.sort_values(by='coefficient', ascending=False).head(10)
top_negative = coef_df.sort_values(by='coefficient', ascending=True).head(10)

print("--- Top 10 Words for Positive Sentiment ---")
print(top_positive)

print("\n--- Top 10 Words for Negative Sentiment ---")
print(top_negative)

--- Top 10 Words for Positive Sentiment ---
                 word  coefficient
181                it     3.834691
0          absolutely     3.801454
426             worth     3.198489
428          worth it     3.194993
2    absolutely worth     3.194993
211            loving     3.184731
249                of     3.174033
7                 and     2.890269
331            so far     2.575774
330                so     2.575774

--- Top 10 Words for Negative Sentiment ---
                  word  coefficient
411               very    -2.869493
255               okay    -2.764855
241                not    -2.565562
20             average    -2.220628
271            overall    -2.072005
431  wouldnt recommend    -2.044715
430            wouldnt    -2.044715
306          recommend    -2.044715
387                 to    -2.010265
56                 but    -2.009847


가중치에 영항을 미친 단어도 긍정부정단어와는 거리가 약간 있어보임

In [19]:
# 1. 평점 기반으로 레이블 재정의 (긍정: 4-5점, 부정: 1-2점, 3점은 중립이므로 제거)
df_refined = df[df['rating'] != 3].copy()
df_refined['label'] = df_refined['rating'].apply(lambda x: 1 if x >= 4 else 0)

# 2. 벡터화 다시 수행 (데이터셋이 바뀜)
X = tfidf.fit_transform(df_refined['cleaned_review'])
y = df_refined['label']

# 3. 데이터셋 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. 모델 재학습
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# 5. 성능 확인
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.94      0.92      3144
           1       0.95      0.94      0.94      4367

    accuracy                           0.94      7511
   macro avg       0.93      0.94      0.93      7511
weighted avg       0.94      0.94      0.94      7511



적절한 수준으로 잘 나온듯?

In [20]:
# TF-IDF 피처 이름(단어들)과 모델의 계수(Coefficient) 매핑
feature_names = tfidf.get_feature_names_out()
coefficients = model.coef_[0]

# 데이터프레임으로 묶기
coef_df = pd.DataFrame({'word': feature_names, 'coefficient': coefficients})

# 긍정에 큰 영향을 주는 단어 Top 10 vs 부정에 큰 영향을 주는 단어 Top 10 확인
top_positive = coef_df.sort_values(by='coefficient', ascending=False).head(10)
top_negative = coef_df.sort_values(by='coefficient', ascending=True).head(10)

print("--- Top 10 Words for Positive Sentiment ---")
print(top_positive)

print("\n--- Top 10 Words for Negative Sentiment ---")
print(top_negative)

--- Top 10 Words for Positive Sentiment ---
                 word  coefficient
0          absolutely     2.354731
181                it     2.080593
2    absolutely worth     1.978024
428          worth it     1.978024
211            loving     1.813723
249                of     1.779316
426             worth     1.672400
238        no regrets     1.489966
379          this one     1.489966
264               one     1.489966

--- Top 10 Words for Negative Sentiment ---
                  word  coefficient
431  wouldnt recommend    -2.264806
430            wouldnt    -2.264806
306          recommend    -2.264806
411               very    -2.222995
412  very disappointed    -1.795596
92        disappointed    -1.795596
316          returning    -1.709778
382          this soon    -1.709778
317     returning this    -1.709778
340               soon    -1.709778


단어의 기조자체는 긍정부정에 확실히 다가간 느낌이지만, 불용어처리를 해야할것 같음 (긍정 가중치 영향 2위가 it)

In [21]:
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# 1. 불용어 리스트 정의 (기본 영어 불용어 + 의미 없는 단어들)
my_stopwords = list(ENGLISH_STOP_WORDS) + ['it', 'this', 'one', 'of', 'and', 'so', 'just', 'like', 'don']

# 2. 평점 기반으로 레이블 재정의 (3점 제외)
df_refined = df[df['rating'] != 3].copy()
df_refined['label'] = df_refined['rating'].apply(lambda x: 1 if x >= 4 else 0)

# 3. TF-IDF 재설정 (불용어 적용)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words=my_stopwords)
X = tfidf.fit_transform(df_refined['cleaned_review'])
y = df_refined['label']

# 4. 데이터셋 분할 및 모델 학습
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# 5. 성능 평가
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

# 6. 가중치 분석 (불용어가 제거된 상태의 상위 단어 확인)
feature_names = tfidf.get_feature_names_out()
coefficients = model.coef_[0]
coef_df = pd.DataFrame({'word': feature_names, 'coefficient': coefficients})

print("\n--- Top 10 Words for Positive Sentiment ---")
print(coef_df.sort_values(by='coefficient', ascending=False).head(10))

print("\n--- Top 10 Words for Negative Sentiment ---")
print(coef_df.sort_values(by='coefficient', ascending=True).head(10))

              precision    recall  f1-score   support

           0       0.91      0.94      0.92      3144
           1       0.95      0.94      0.94      4367

    accuracy                           0.94      7511
   macro avg       0.93      0.94      0.93      7511
weighted avg       0.94      0.94      0.94      7511


--- Top 10 Words for Positive Sentiment ---
                 word  coefficient
0          absolutely     2.910773
164            loving     2.658057
240           regrets     2.638402
241    regrets buying     2.638402
2    absolutely worth     2.233523
166        loving far     2.181827
103               far     2.181827
308             worth     1.816468
110             feels     1.806027
42             buying     1.696118

--- Top 10 Words for Negative Sentiment ---
                  word  coefficient
167               mark    -3.126087
77        disappointed    -2.896626
311            wouldnt    -2.132854
312  wouldnt recommend    -2.132854
232          recom

긍정 평가에 regrets buying 같은 단어가있는건 아직 이상함
N-Gram 의 한계일지도? No를 불용어 처리해버려서 삭제되었다면 말이 안되지는 않지만...